# S4b.3 — Multi-Query Retrieval

This notebook verifies the multi-query retrieval service:
- Query variation generation via LLM
- Parallel hybrid search across variations
- Result deduplication by chunk_id
- RRF fusion scoring
- Graceful fallback on failures

In [1]:
import sys
sys.path.insert(0, "../..")

from src.config import MultiQuerySettings, Settings

# Verify settings exist and have correct defaults
settings = MultiQuerySettings()
assert settings.enabled is True
assert settings.num_queries == 3
assert settings.temperature == 0.7
assert settings.max_tokens == 300
assert settings.rrf_k == 60

root = Settings()
assert hasattr(root, 'multi_query')
assert root.multi_query.enabled is True

print("\n✓ MultiQuerySettings configured with correct defaults")
print(f"  num_queries={settings.num_queries}, temperature={settings.temperature}, rrf_k={settings.rrf_k}")


✓ MultiQuerySettings configured with correct defaults
  num_queries=3, temperature=0.7, rrf_k=60


In [2]:
from src.services.retrieval.multi_query import MultiQueryService, MultiQueryResult

# Verify classes exist and are importable
assert MultiQueryService is not None
assert MultiQueryResult is not None

# Check MultiQueryResult fields
r = MultiQueryResult()
assert r.original_query == ""
assert r.generated_queries == []
assert r.results == []

print("\n✓ MultiQueryService and MultiQueryResult importable")
print(f"  MultiQueryResult fields: original_query, generated_queries, results")


✓ MultiQueryService and MultiQueryResult importable
  MultiQueryResult fields: original_query, generated_queries, results


In [3]:
from src.services.retrieval.factory import create_multi_query_service
from unittest.mock import AsyncMock, MagicMock

# Verify factory creates service correctly
service = create_multi_query_service(
    settings=MultiQuerySettings(),
    llm_provider=AsyncMock(),
    embeddings_client=AsyncMock(),
    opensearch_client=MagicMock(),
)
assert isinstance(service, MultiQueryService)

print("\n✓ Factory function creates MultiQueryService correctly")


✓ Factory function creates MultiQueryService correctly


In [4]:
# Test query variation parsing (internal method)
variations_numbered = MultiQueryService._parse_variations(
    "1. How do transformer models work?\n"
    "2. What is the architecture of transformers?\n"
    "3. Self-attention in NLP models"
)
assert len(variations_numbered) == 3
assert variations_numbered[0] == "How do transformer models work?"

variations_bullets = MultiQueryService._parse_variations(
    "- First variation\n- Second variation\n- Third variation"
)
assert len(variations_bullets) == 3

# Empty/whitespace lines are skipped
variations_with_gaps = MultiQueryService._parse_variations(
    "1. First\n\n2. Second\n   \n3. Third"
)
assert len(variations_with_gaps) == 3

print("\n✓ Query variation parsing works for numbered lists, bullets, and gaps")
print(f"  Parsed: {variations_numbered}")


✓ Query variation parsing works for numbered lists, bullets, and gaps
  Parsed: ['How do transformer models work?', 'What is the architecture of transformers?', 'Self-attention in NLP models']


In [5]:
from src.services.llm.provider import LLMResponse, UsageMetadata
from src.schemas.api.search import SearchHit

# Set up mocks for full pipeline test
mock_llm = AsyncMock()
mock_llm.generate = AsyncMock(return_value=LLMResponse(
    text="1. How do transformer neural networks work?\n2. Architecture of attention-based models\n3. Self-attention in NLP",
    model="gemini-2.0-flash",
    provider="gemini",
    usage=UsageMetadata(prompt_tokens=50, completion_tokens=40, total_tokens=90),
))

mock_embeddings = AsyncMock()
mock_embeddings.embed_query = AsyncMock(return_value=[0.1] * 1024)

mock_opensearch = MagicMock()
mock_opensearch.search_chunks_hybrid = MagicMock(return_value={
    "total": 2,
    "hits": [
        {"arxiv_id": "1706.03762", "title": "Attention Is All You Need",
         "authors": ["Vaswani"], "abstract": "...", "pdf_url": "...",
         "chunk_text": "...", "chunk_id": "1706.03762_chunk_0",
         "section_title": "Introduction", "score": 0.95, "highlights": {}},
        {"arxiv_id": "1810.04805", "title": "BERT",
         "authors": ["Devlin"], "abstract": "...", "pdf_url": "...",
         "chunk_text": "...", "chunk_id": "1810.04805_chunk_0",
         "section_title": "Model", "score": 0.88, "highlights": {}},
    ],
})

svc = MultiQueryService(
    settings=MultiQuerySettings(),
    llm_provider=mock_llm,
    embeddings_client=mock_embeddings,
    opensearch_client=mock_opensearch,
)

result = await svc.retrieve_with_multi_query("What are transformers?")

assert isinstance(result, MultiQueryResult)
assert result.original_query == "What are transformers?"
assert len(result.generated_queries) == 3
assert len(result.results) == 2  # 2 unique chunks across all queries
assert mock_llm.generate.call_count == 1
assert mock_embeddings.embed_query.call_count == 3  # one per variation
assert mock_opensearch.search_chunks_hybrid.call_count == 3

print("\n✓ Full multi-query pipeline works end-to-end")
print(f"  Original: {result.original_query}")
print(f"  Variations: {result.generated_queries}")
print(f"  Results: {len(result.results)} unique chunks")
for hit in result.results:
    print(f"    - {hit.title} ({hit.chunk_id}) score={hit.score:.4f}")


✓ Full multi-query pipeline works end-to-end
  Original: What are transformers?
  Variations: ['How do transformer neural networks work?', 'Architecture of attention-based models', 'Self-attention in NLP']
  Results: 2 unique chunks
    - Attention Is All You Need (1706.03762_chunk_0) score=0.0492
    - BERT (1810.04805_chunk_0) score=0.0484


In [6]:
# Test RRF fusion math
# With k=60: rank 1 -> 1/61, rank 2 -> 1/62
# Doc appearing at rank 1 in 3 queries: 3 * 1/61 = 0.04918
# Doc appearing at rank 1 in 2, rank 2 in 1: 2/61 + 1/62 = 0.04893

rrf_rank1_x3 = 3 * (1.0 / 61)
rrf_rank1_x2_rank2_x1 = 2 * (1.0 / 61) + (1.0 / 62)

assert rrf_rank1_x3 > rrf_rank1_x2_rank2_x1, "Higher rank consistency should yield higher RRF score"

print("\n✓ RRF fusion math verified")
print(f"  3x rank 1: {rrf_rank1_x3:.5f}")
print(f"  2x rank 1 + 1x rank 2: {rrf_rank1_x2_rank2_x1:.5f}")


✓ RRF fusion math verified
  3x rank 1: 0.04918
  2x rank 1 + 1x rank 2: 0.04892


In [7]:
# Test graceful fallback when LLM fails
mock_llm_fail = AsyncMock()
mock_llm_fail.generate = AsyncMock(side_effect=Exception("LLM unavailable"))

svc_fallback = MultiQueryService(
    settings=MultiQuerySettings(),
    llm_provider=mock_llm_fail,
    embeddings_client=mock_embeddings,
    opensearch_client=mock_opensearch,
)

# Reset call counts
mock_embeddings.embed_query.reset_mock()
mock_opensearch.search_chunks_hybrid.reset_mock()

result_fallback = await svc_fallback.retrieve_with_multi_query("What are transformers?")

assert result_fallback.generated_queries == ["What are transformers?"]
assert len(result_fallback.results) > 0
assert mock_embeddings.embed_query.call_count == 1  # Only original query
assert mock_opensearch.search_chunks_hybrid.call_count == 1

print("\n✓ Graceful fallback works when LLM fails")
print(f"  Used original query: {result_fallback.generated_queries}")
print(f"  Still returned {len(result_fallback.results)} results")

Multi-query generation failed, using original query
Traceback (most recent call last):
  File "/Users/nishantgaurav/Project/PaperAlchemy/notebooks/specs/../../src/services/retrieval/multi_query.py", line 76, in generate_query_variations
    response = await self._llm.generate(
               ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/lib/python3.12/unittest/mock.py", line 2282, in _execute_mock_call
    raise effect
Exception: LLM unavailable



✓ Graceful fallback works when LLM fails
  Used original query: ['What are transformers?']
  Still returned 2 results
